# Day 2 | NumPy and Pandas Core

This notebook supports day two of the module.

## Focus
- loading multiple files with deliberate dtypes
- cleaning key fields
- joins across customers, accounts, and transactions
- groupby KPIs and monthly summaries
- lightweight data quality reporting

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/day2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(DATA_DIR / "customers.csv", dtype={"customer_id": "string"})
accounts = pd.read_csv(
    DATA_DIR / "accounts.csv",
    dtype={"account_id": "string", "customer_id": "string"},
)
transactions = pd.read_csv(
    DATA_DIR / "transactions.csv",
    dtype={"account_id": "string", "branch_id": "string"},
)
branches = pd.read_csv(DATA_DIR / "branches.csv", dtype={"branch_id": "string"})

customers["onboarding_date"] = pd.to_datetime(customers["onboarding_date"], errors="coerce")
accounts["open_date"] = pd.to_datetime(accounts["open_date"], errors="coerce")
transactions["txn_ts"] = pd.to_datetime(transactions["txn_ts"], errors="coerce")
transactions["amount_sar"] = pd.to_numeric(transactions["amount_sar"], errors="coerce")
transactions["fee_sar"] = pd.to_numeric(transactions["fee_sar"], errors="coerce")

customers["region_clean"] = customers["region"].str.strip().str.title()
transactions["channel_clean"] = transactions["channel"].fillna("UNKNOWN").str.strip().str.upper()
transactions.head()

In [ ]:
duplicate_customers = customers[customers.duplicated("customer_id", keep=False)].copy()
invalid_regions = customers.loc[
    ~customers["region_clean"].isin(["Riyadh", "Makkah", "Eastern", "Madinah", "Asir"]),
    ["customer_id", "region", "region_clean"],
]

analysis_df = (
    transactions.merge(accounts[["account_id", "customer_id", "product_family"]], on="account_id", how="left")
    .merge(customers[["customer_id", "region_clean", "segment"]], on="customer_id", how="left")
    .merge(branches[["branch_id", "region", "city"]], on="branch_id", how="left")
)

posted = analysis_df[analysis_df["status"] == "POSTED"].copy()
branch_kpi = (
    posted.groupby(["region", "branch_id"], dropna=False)
    .agg(
        txn_count=("txn_id", "count"),
        total_fee_sar=("fee_sar", "sum"),
        avg_ticket_sar=("amount_sar", "mean"),
        active_customers=("customer_id", "nunique"),
    )
    .reset_index()
    .sort_values(["region", "total_fee_sar"], ascending=[True, False])
)

monthly = (
    posted.assign(month=posted["txn_ts"].dt.to_period("M").astype(str))
    .groupby(["month", "region"])
    .agg(txn_count=("txn_id", "count"), total_fee_sar=("fee_sar", "sum"))
    .reset_index()
)

checks = pd.DataFrame(
    [
        {"check": "duplicate customer_id", "status": "fail" if len(duplicate_customers) else "pass", "count": len(duplicate_customers)},
        {"check": "invalid region values", "status": "fail" if len(invalid_regions) else "pass", "count": len(invalid_regions)},
        {"check": "missing transaction amount", "status": "fail" if posted["amount_sar"].isna().sum() else "pass", "count": int(posted["amount_sar"].isna().sum())},
    ]
)

branch_kpi.to_csv(OUTPUT_DIR / "branch_kpi.csv", index=False)
monthly.to_csv(OUTPUT_DIR / "monthly_metrics.csv", index=False)
checks.to_csv(OUTPUT_DIR / "dq_report.csv", index=False)

branch_kpi.head()

## Day 2 Exercises

### Exercise C
- inspect duplicates and invalid region values in `customers`
- export a `dq_report.csv`

### Exercise D
- produce a branch KPI table with `txn_count`, `total_fee_sar`, and `avg_ticket_sar`
- extend with `active_customers` if time allows

### Exercise E
- summarise product uptake by region
- discuss denominator choices before presenting results